### Imports

In [1]:
import torch
import openai
import pandas as pd
import os
from bertopic import BERTopic
from umap import UMAP
from hdbscan import HDBSCAN
from sentence_transformers import SentenceTransformer
from sklearn.feature_extraction.text import CountVectorizer
from bertopic.representation import OpenAI
from tqdm.auto import tqdm
from dotenv import load_dotenv
load_dotenv()

True

### Load Data

- Downloaded from PRAW
- r/cybersecurity (global issues, surveillance, intersection of CS and social issues)
- posts from 2024-01-01 to 2026-04-06 6:20 PM

In [2]:
posts_df = pd.read_json('r_cybersecurity_posts.jsonl', lines=True)
posts_df

,_meta,all_awardings,allow_live_comments,approved_at_utc,approved_by,archived,author,author_flair_background_color,author_flair_css_class,author_flair_richtext,...,url_overridden_by_dest,crosspost_parent,crosspost_parent_list,poll_data,author_cakeday,location_lat,location_long,location_name,websocket_url,outbound_link
0,{'note': 'no_2nd_retrieval'},[],False,NaN,NaN,False,AutoModerator,None,None,[],...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,{'note': 'no_2nd_retrieval'},[],False,NaN,NaN,False,ibeawuchi,None,None,[],...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,{'note': 'no_2nd_retrieval'},[],False,NaN,NaN,False,PanicInTheKernel,None,None,[],...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,{'note': 'no_2nd_retrieval'},[],False,NaN,NaN,False,Warm_Ad_7120,None,None,[],...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,{'note': 'no_2nd_retrieval'},[],False,NaN,NaN,False,NaiveJello1914,None,None,[],...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
95232,NaN,[],False,NaN,NaN,False,Wild-Rest-3627,None,None,[],...,https://youtube.com/shorts/VYvwsLGi9RI?si=-00Z...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,wss://k8s-lb.wss.redditmedia.com/link/1sdw3te?...,{'url': 'https://out.reddit.com/t3_1sdw3te?url...
95233,NaN,[],False,NaN,NaN,False,NISMO1968,None,None,[],...,https://arstechnica.com/security/2026/04/new-r...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
95234,NaN,[],False,NaN,NaN,False,Astofol760,None,None,[],...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
95235,NaN,[],False,NaN,NaN,False,NISMO1968,None,None,[],...,https://www.scworld.com/news/axios-maintainers...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,wss://k8s-lb.wss.redditmedia.com/link/1sdwya8?...,{'url': 'https://out.reddit.com/t3_1sdwya8?url...


In [3]:
comments_df = pd.read_json('r_cybersecurity_comments.jsonl', lines=True)
comments_df

,_meta,all_awardings,approved_at_utc,approved_by,archived,associated_award,author,author_flair_background_color,author_flair_css_class,author_flair_richtext,...,treatment_tags,unrepliable_reason,ups,user_reports,author_cakeday,editable,body_sha1,nest_level,profile_img,profile_over_18
0,{'retrieved_2nd_on': 1704196851},[],NaN,NaN,False,NaN,AutoModerator,None,None,[],...,[],NaN,1,[],NaN,NaN,NaN,NaN,NaN,NaN
1,"{'is_edited': True, 'retrieved_2nd_on': 170419...",[],NaN,NaN,False,NaN,Xzarkuun,None,None,[],...,[],NaN,1,[],NaN,NaN,NaN,NaN,NaN,NaN
2,{'retrieved_2nd_on': 1704196970},[],NaN,NaN,False,NaN,MSXzigerzh0,None,None,[],...,[],NaN,2,[],NaN,NaN,NaN,NaN,NaN,NaN
3,{'retrieved_2nd_on': 1704197151},[],NaN,NaN,False,NaN,jwrig,None,None,[],...,[],NaN,1,[],NaN,NaN,NaN,NaN,NaN,NaN
4,{'retrieved_2nd_on': 1704197214},[],NaN,NaN,False,NaN,me_z,#373c3f,None,[],...,[],NaN,5,[],NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
695184,NaN,[],NaN,NaN,False,NaN,makeiteasy_24,None,None,[],...,[],NaN,1,[],NaN,NaN,NaN,NaN,https://styles.redditmedia.com/t5_h54npz/style...,0.0
695185,NaN,[],NaN,NaN,False,NaN,1littlenapoleon,None,None,[],...,[],NaN,1,[],NaN,NaN,NaN,NaN,https://styles.redditmedia.com/t5_ep18l/styles...,0.0
695186,NaN,[],NaN,NaN,False,NaN,Forward_Switch1015,None,None,[],...,[],NaN,1,[],NaN,NaN,NaN,NaN,https://i.redd.it/snoovatar/avatars/f42d464c-d...,1.0
695187,NaN,[],NaN,NaN,False,NaN,LastFisherman373,None,None,[],...,[],NaN,1,[],NaN,NaN,NaN,NaN,https://i.redd.it/snoovatar/avatars/3afc41ce-6...,0.0


#### Data Cleaning
- Remove posts and comments by known reddit bots
- Remove posts and comments that were marked as '\[removed\]' or '\[deleted\]'
- Remove posts with the flairs: 'Career Questions & Discussion', 'Tutorial' as they are not relevant to the social positions we wish to study.\
- Remove posts and comments by '\[deleted\]' authors

In [4]:
drop_users = {'AutoModerator', 'RemindMeBot', '[deleted]'}
drop_content = {'[removed]', '[deleted]'}
drop_flairs = {'Career Questions & Discussion', 'Tutorial'}
drop_titles = {'[ Removed by moderator ]', '[ Removed by Reddit ]'}

Posts

In [5]:
clean_posts = posts_df[~posts_df['author'].isin(drop_users)].copy()
clean_posts = clean_posts[~clean_posts['link_flair_text'].isin(drop_flairs)]

clean_posts['selftext'] = clean_posts['selftext'].fillna('')
clean_posts = clean_posts[~clean_posts['selftext'].isin(drop_content)]

clean_posts['title'] = clean_posts['title'].fillna('')
clean_posts = clean_posts[~clean_posts['title'].isin(drop_titles)]

clean_posts

,_meta,all_awardings,allow_live_comments,approved_at_utc,approved_by,archived,author,author_flair_background_color,author_flair_css_class,author_flair_richtext,...,url_overridden_by_dest,crosspost_parent,crosspost_parent_list,poll_data,author_cakeday,location_lat,location_long,location_name,websocket_url,outbound_link
2,{'note': 'no_2nd_retrieval'},[],False,NaN,NaN,False,PanicInTheKernel,None,None,[],...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,{'note': 'no_2nd_retrieval'},[],False,NaN,NaN,False,Warm_Ad_7120,None,None,[],...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
9,{'note': 'no_2nd_retrieval'},[],False,NaN,NaN,False,elitetek,None,None,[],...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
10,{'note': 'no_2nd_retrieval'},[],False,NaN,NaN,False,Itchy_Sprinkles,None,None,[],...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
14,{'note': 'no_2nd_retrieval'},[],False,NaN,NaN,False,zer0pRiME-X,None,None,[],...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
95227,NaN,[],False,NaN,NaN,False,Big-Engineering-9365,None,None,[],...,https://threatroad.substack.com/p/controls-sha...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
95232,NaN,[],False,NaN,NaN,False,Wild-Rest-3627,None,None,[],...,https://youtube.com/shorts/VYvwsLGi9RI?si=-00Z...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,wss://k8s-lb.wss.redditmedia.com/link/1sdw3te?...,{'url': 'https://out.reddit.com/t3_1sdw3te?url...
95233,NaN,[],False,NaN,NaN,False,NISMO1968,None,None,[],...,https://arstechnica.com/security/2026/04/new-r...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
95235,NaN,[],False,NaN,NaN,False,NISMO1968,None,None,[],...,https://www.scworld.com/news/axios-maintainers...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,wss://k8s-lb.wss.redditmedia.com/link/1sdwya8?...,{'url': 'https://out.reddit.com/t3_1sdwya8?url...


Comments

In [6]:
clean_comments = comments_df[~comments_df['author'].isin(drop_users)].copy()

clean_comments['body'] = clean_comments['body'].fillna('')
clean_comments = clean_comments[~clean_comments['body'].isin(drop_content)]

# remove all comments from posts with irrelevant flair
valid_post_ids = "t3_" + clean_posts['id'].astype(str)
clean_comments = clean_comments[clean_comments['link_id'].isin(valid_post_ids)]

clean_comments

,_meta,all_awardings,approved_at_utc,approved_by,archived,associated_award,author,author_flair_background_color,author_flair_css_class,author_flair_richtext,...,treatment_tags,unrepliable_reason,ups,user_reports,author_cakeday,editable,body_sha1,nest_level,profile_img,profile_over_18
29,{'retrieved_2nd_on': 1704199395},[],NaN,NaN,False,NaN,anevilbor,#373c3f,None,[],...,[],NaN,20,[],NaN,NaN,NaN,NaN,NaN,NaN
44,{'retrieved_2nd_on': 1704200329},[],NaN,NaN,False,NaN,MaxHedrome,None,None,[],...,[],NaN,10,[],NaN,NaN,NaN,NaN,NaN,NaN
73,{'retrieved_2nd_on': 1704202724},[],NaN,NaN,False,NaN,Upstairs_Lie_9379,None,None,[],...,[],NaN,1,[],NaN,NaN,NaN,NaN,NaN,NaN
78,{'retrieved_2nd_on': 1704202857},[],NaN,NaN,False,NaN,Shaaaaazam,None,None,[],...,[],NaN,4,[],NaN,NaN,NaN,NaN,NaN,NaN
83,{'retrieved_2nd_on': 1704204470},[],NaN,NaN,False,NaN,alara_zero,None,None,[],...,[],NaN,1,[],NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
695175,NaN,[],NaN,NaN,False,NaN,blipojones,None,None,[],...,[],NaN,1,[],NaN,NaN,NaN,NaN,https://i.redd.it/snoovatar/avatars/97d190a3-8...,0.0
695177,NaN,[],NaN,NaN,False,NaN,blipojones,None,None,[],...,[],NaN,1,[],NaN,NaN,NaN,NaN,https://i.redd.it/snoovatar/avatars/97d190a3-8...,0.0
695179,NaN,[],NaN,NaN,False,NaN,CammKelly,None,None,[],...,[],NaN,1,[],NaN,NaN,NaN,NaN,https://i.redd.it/snoovatar/avatars/32d5d73b-c...,1.0
695181,NaN,[],NaN,NaN,False,NaN,todbatx,None,None,[],...,[],NaN,1,[],NaN,NaN,NaN,NaN,https://i.redd.it/snoovatar/avatars/4a49f053-d...,0.0


### 1.1 Aggragate Stats

In [7]:
print('Total Post Count from 2024-01-01 to 2026-04-06:', len(clean_posts))
print('Total Comment Count from 2024-01-01 to 2026-04-06:', len(clean_comments))

post_authors = set(clean_posts['author'].dropna())
comment_authors = set(clean_comments['author'].dropna())
unique_users = post_authors.union(comment_authors)

print('Total Unique User Count from 2024-01-01 to 2026-04-06:', len(unique_users))

Total Post Count from 2024-01-01 to 2026-04-06: 37350
Total Comment Count from 2024-01-01 to 2026-04-06: 450871
Total Unique User Count from 2024-01-01 to 2026-04-06: 91916


In [8]:
print('Average Post Score from 2024-01-01 to 2026-04-06:', clean_posts['score'].mean())
print('Median Post Score from 2024-01-01 to 2026-04-06:', clean_posts['score'].median())
print('Maximum Post Score from 2024-01-01 to 2026-04-06:', clean_posts['score'].max())

print()

print('Average Comment Score from 2024-01-01 to 2026-04-06:', clean_comments['score'].mean())
print('Median Comment Score from 2024-01-01 to 2026-04-06:', clean_comments['score'].median())
print('Maximum Comment Score from 2024-01-01 to 2026-04-06:', clean_comments['score'].max())

Average Post Score from 2024-01-01 to 2026-04-06: 29.402302543507364
Median Post Score from 2024-01-01 to 2026-04-06: 1.0
Maximum Post Score from 2024-01-01 to 2026-04-06: 8184

Average Comment Score from 2024-01-01 to 2026-04-06: 6.692000594405052
Median Comment Score from 2024-01-01 to 2026-04-06: 2.0
Maximum Comment Score from 2024-01-01 to 2026-04-06: 2343


### Prepare NLP Setup

- Refactor databases to something easier to parse for subsequent tasks
    - Create a merged column with post body + title in posts dataframe
    - Only preserve this column, along with author, creation_utc (date), and score columns
    - Repeat this with column database
    - Concatanate both into a full database
- Use BERTopic for assigning topics with its outlined [best practices](https://maartengr.github.io/BERTopic/getting_started/best_practices/best_practices.html)
    - Prevent stochastic behaviour (replicability)
    - Fix number of topics (10)
    - Use MTEB to find most appropriate embeddings for 2026
    - Use BERTopic's [integration with LLMs](https://maartengr.github.io/BERTopic/getting_started/representation/llm.html#truncating-documents) to generate topic labels and summaries.
        - Using Google's gemma-4-31b-it for **Topic Labelling and Description** based on keywords and documents, routed via OpenRouter

Refactoring

In [9]:
p_df = clean_posts[['author', 'created_utc', 'id', 'score']].copy()
p_df['text'] = clean_posts['title'].fillna('') + ": " + clean_posts['selftext'].fillna('')
p_df['type'] = 'post'
p_df['parent_post_id'] = "t3_" + p_df['id'].astype(str) 

c_df = clean_comments[['author', 'created_utc', 'link_id', 'score']].copy()
c_df['text'] = clean_comments['body'].fillna('')
c_df['type'] = 'comment'
c_df['parent_post_id'] = c_df['link_id']

full_df = pd.concat([
    p_df[['author', 'text', 'created_utc', 'type', 'parent_post_id', 'score']],
    c_df[['author', 'text', 'created_utc', 'type', 'parent_post_id', 'score']]
], ignore_index=True)

full_df['date'] = pd.to_datetime(full_df['created_utc'], unit='s')

full_df

,author,text,created_utc,type,parent_post_id,score,date
0,PanicInTheKernel,Canary tokens: I’m a fan of canarytokens.org a...,1704067769,post,t3_18vkn27,1,2024-01-01 00:09:29
1,Warm_Ad_7120,Evaluation of non-legitimate Scheduled Tasks p...,1704068145,post,t3_18vkr9q,1,2024-01-01 00:15:45
2,elitetek,"Penetration testing?: Hey guys, I’m about to p...",1704072783,post,t3_18vm4ym,1,2024-01-01 01:33:03
3,Itchy_Sprinkles,I'd like to become a soc analyst in the future...,1704072957,post,t3_18vm6ud,3,2024-01-01 01:35:57
4,zer0pRiME-X,Possibly the most sophisticated exploit ever: ...,1704078446,post,t3_18vnpac,1,2024-01-01 03:07:26
...,...,...,...,...,...,...,...
488216,blipojones,"Ye i'm looking to ""go local"", like IRL face to...",1775477923,comment,t3_1sdgj6n,1,2026-04-06 12:18:43
488217,blipojones,ye this is exactly the fuzzy area i was wonder...,1775478138,comment,t3_1sdgj6n,1,2026-04-06 12:22:18
488218,CammKelly,Yup. Until then its a really awkward discussio...,1775478356,comment,t3_1sd7rtm,1,2026-04-06 12:25:56
488219,todbatx,https://www.runzero.com/resources/deciphering-...,1775478536,comment,t3_1safq3a,1,2026-04-06 12:28:56


NLP

In [10]:
# replicability
umap_model = UMAP(n_neighbors=15, n_components=5, min_dist=0.0, metric='cosine', 
                  random_state=42) # random state is the main setting, the rest are defaults needed
                                   # for BERTopic

# topic count reduction (internal algo)
hdbscan_model = HDBSCAN(min_cluster_size=300, # same idea as UMAP here
                        metric='euclidean', cluster_selection_method='eom', prediction_data=True)

vectorizer_model = CountVectorizer(stop_words="english", min_df=2, ngram_range=(1, 2))
# remove english stop words, look at bigrams for topics, + BERTopic default

client = openai.OpenAI(base_url="https://openrouter.ai/api/v1", api_key=os.getenv("OPENROUTER_API_KEY"))

prompt_label = """
I have a topic that contains the following documents:
[DOCUMENTS]
The topic is described by the following keywords: [KEYWORDS]

Based on the information above, extract a short topic label in the following format:
topic: <topic label>
"""
label_model = OpenAI(client, model='google/gemma-4-31b-it:free', prompt=prompt_label, delay_in_seconds=3)
# gemma4-31b for label

prompt_summary = """
I have a topic that is described by the following keywords: [KEYWORDS]
In this topic, the following documents are a small but representative subset of all documents in the topic:
[DOCUMENTS]

Based on the information above, please give a short description of this topic in the following format:
topic: <description>
"""
summary_model = OpenAI(client, model='google/gemma-4-31b-it:free', prompt=prompt_summary, delay_in_seconds=3)
# gemma4-31b for summary

representation_model = {
   "Label": label_model,
   "Summary": summary_model,
}

device = "mps" if torch.backends.mps.is_available() else "cpu"
print(f"Using device: {device}")

embedding_model = SentenceTransformer("all-MiniLM-L6-v2", device=device)
embeddings = embedding_model.encode(full_df['text'].to_list(), show_progress_bar=True, batch_size=64)

topic_model = BERTopic(
    embedding_model=embedding_model,
    umap_model=umap_model,
    hdbscan_model=hdbscan_model,
    vectorizer_model=vectorizer_model,
    representation_model=representation_model,

    nr_topics=11, # topic count reduction (final topic count)
    verbose=True
)

Using device: mps


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Batches:   0%|          | 0/7629 [00:00<?, ?it/s]

In [11]:
topics, _ = topic_model.fit_transform(full_df['text'].to_list(), embeddings)

2026-04-10 01:27:36,960 - BERTopic - Dimensionality - Fitting the dimensionality reduction algorithm
2026-04-10 01:56:10,429 - BERTopic - Dimensionality - Completed ✓
2026-04-10 01:56:10,433 - BERTopic - Cluster - Start clustering the reduced embeddings
2026-04-10 01:57:11,476 - BERTopic - Cluster - Completed ✓
2026-04-10 01:57:11,479 - BERTopic - Representation - Extracting topics using c-TF-IDF for topic reduction.
2026-04-10 01:57:27,363 - BERTopic - Representation - Completed ✓
2026-04-10 01:57:27,376 - BERTopic - Topic reduction - Reducing number of topics
/Users/rupsharray/Documents/Ashoka/Academics/Semester 6/NLP/Project/SecInfo/reddit_env/lib/python3.10/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: divide by zero encountered in matmul
  ret = a @ b
/Users/rupsharray/Documents/Ashoka/Academics/Semester 6/NLP/Project/SecInfo/reddit_env/lib/python3.10/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: overflow encountered in matmul
  ret = a @ b
/Users/rupsh

In [14]:
topic_data = topic_model.get_topic_info()
topic_data.to_parquet('cybersec_topic_data.pkl')

In [15]:
full_df_labeled = topic_model.get_document_info(full_df['text'].to_list())
full_df_labeled.to_parquet('cybersec_full_labeled_data.pkl')

In [16]:
topic_model.get_topic_info()

,Topic,Count,Name,Representation,Label,Summary,Representative_Docs
0,-1,256823,-1_security_like_just_https,"[security, like, just, https, don, people, cyb...",[Cybersecurity Career Advice and Entry-Level G...,[Career advice and professional development wi...,"[You told everyone your experience, and then a..."
1,0,128271,0_security_just_data_like,"[security, just, data, like, ai, use, don, peo...",[Cybersecurity Risks and Professional Perspect...,"[Cybersecurity risk management, professional d...",[SCORM Dangers: I am new to the r/cybersecurit...
2,1,47034,1_crowdstrike_thanks_thank_just,"[crowdstrike, thanks, thank, just, like, don, ...",[CrowdStrike Reputation and Market Impact],[Discussions and opinions regarding CrowdStrik...,"[😆😆 I like that!, Is CrowdStrike Invisible?! -..."
3,2,46106,2_just_job_like_work,"[just, job, like, work, don, risk, security, e...",[Cybersecurity Compliance and GRC Certifications],"[GRC (Governance, Risk, and Compliance) career...",[Best risk management tool for low-maturity ri...
4,3,5308,3_linux_python_kali_tools,"[linux, python, kali, tools, use, windows, too...",[Operating Systems & Programming Tools],[Learning and using Linux and Python for secur...,"[Linux for me, Linux, Linux+?]"
5,4,1468,4_quantum_hash_encryption_quantum computing,"[quantum, hash, encryption, quantum computing,...",[Post-Quantum Cryptography and Quantum Computi...,[Post-Quantum Cryptography (PQC) and the trans...,"[quantum computing is a meme, That's a great q..."
6,5,835,5_lmao_following fascinating_yikes lmao_rosé,"[lmao, following fascinating, yikes lmao, rosé...",[casual reactions and slang],[Informal social interactions and emotional re...,"[Rosé 😉, 🤣🤣🤣🤣. Clearly you don’t understand., ..."
7,6,741,6_solarwinds_fortinet_solar_fortigate,"[solarwinds, fortinet, solar, fortigate, https...",[SolarWinds and Fortinet security concerns],[Security vulnerabilities and supply chain att...,"[SolarWinds?, Solarwinds?, Anything made by So..."
8,7,651,7_que_la_en_para,"[que, la, en, para, el, se, le, es, est, por]",[Cybersecurity and Artificial Intelligence (AI)],"[Cybersecurity, ethical hacking, and the integ...",[Perdí oportunidades laborales por no saber in...
9,8,640,8_pdf_printer_print_printers,"[pdf, printer, print, printers, adobe, hp, fil...",[Cybersecurity Risks in PDF Software and Print...,[Security vulnerabilities and threats associat...,"[Brother printer., PDF editor software has bec..."
